# P2-01 · Indexación con Chunking Semántico
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa el **Chunking Semántico** usando `SemanticChunker` de LangChain
En lugar de dividir el texto por tamaño fijo, agrupa fragmentos por coherencia temática.

## Diferencia clave vs Práctica 1
| Práctica 1 | Práctica 2 |
|---|---|
| `RecursiveCharacterTextSplitter` (tamaño fijo) | `SemanticChunker` (por significado) |
| Chunks de 1000 chars con 200 overlap | Chunks variables según coherencia semántica |
| Rápido pero puede cortar ideas a la mitad | Preserva unidades semánticas completas |

## 0 · Instalación y configuración

In [ ]:
# Instalar dependencias adicionales para Práctica 2
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    langchain-experimental==0.3.4 \
    langchain-groq==0.2.3 \
    faiss-cpu==1.9.0 \
    arxiv==2.1.3 \
    pypdf==5.1.0 \
    sentence-transformers==3.3.1 \
    rdflib==7.1.1 \
    SPARQLWrapper==2.0.0

print('✅ Dependencias instaladas')

In [ ]:
import os
from google.colab import userdata

# ── Claves API ────────────────────────────────────────────────────────────
os.environ['GOOGLE_API_KEY']      = userdata.get('GOOGLE_API_KEY')
os.environ['GROQ_API_KEY']        = userdata.get('GROQ_API_KEY')
os.environ['LANGCHAIN_API_KEY']   = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']   = 'RAG-KnowledgeGraph-P2'
os.environ['TAVILY_API_KEY']      = userdata.get('TAVILY_API_KEY')  # para búsqueda web (fallback)

print('✅ Claves API configuradas')
print(f'   LangSmith project: {os.environ["LANGCHAIN_PROJECT"]}')

In [ ]:
# ── Imports principales ───────────────────────────────────────────────────
import json, pickle, time, hashlib
from pathlib import Path
from typing import List, Dict, Any

# LangChain
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_community.document_loaders import PyPDFLoader

print('✅ Imports completados')

## 1 · Montar Drive y verificar corpus

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Rutas del proyecto ────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/RAG_P2')
CORPUS_DIR  = BASE_DIR / 'corpus'         # PDFs del corpus (reutilizados de P1)
INDEX_DIR   = BASE_DIR / 'indices_p2'     # Índices FAISS con chunking semántico
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Verificar corpus
pdfs = list(CORPUS_DIR.glob('*.pdf'))
print(f'✅ Corpus cargado: {len(pdfs)} documentos PDF')
for pdf in pdfs[:5]:
    print(f'   • {pdf.name}')
if len(pdfs) > 5:
    print(f'   ... y {len(pdfs)-5} más')

## 2 · Inicializar modelos de embeddings

In [ ]:
# ── Modelo de embeddings (Google text-embedding-004) ─────────────────────
# Se usa el mismo modelo que en P1 para mantener compatibilidad
embeddings_model = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004',
    task_type='retrieval_document'
)

print('✅ Modelo de embeddings inicializado: text-embedding-004')

# Test rápido
test_emb = embeddings_model.embed_query('test biomédico')
print(f'   Dimensión del embedding: {len(test_emb)}')

## 3 · SemanticChunker — Chunking Semántico

El `SemanticChunker` divide el texto en chunks basándose en la **similitud semántica** entre frases consecutivas.
Cuando la similitud cae por debajo de un umbral, se considera un límite de chunk.

### Estrategias de umbral disponibles:
| Estrategia | Descripción |
|---|---|
| `percentile` | Umbral = percentil X de las distancias |
| `standard_deviation` | Umbral = media + N * desviación estándar |
| `interquartile` | Umbral basado en rango intercuartílico |

In [ ]:
# ── Configurar SemanticChunker ────────────────────────────────────────────
# breakpoint_threshold_type='percentile': divide donde la distancia semántica
# entre frases consecutivas supera el percentil 95
semantic_chunker = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type='percentile',   # estrategia de umbral
    breakpoint_threshold_amount=95            # percentil 95 → chunks más grandes y coherentes
)

print('✅ SemanticChunker configurado')
print('   Estrategia: percentile (95)')
print('   Comportamiento: divide el texto cuando hay un salto semántico significativo')

## 4 · Cargar y procesar documentos

In [ ]:
def load_and_chunk_document(pdf_path: Path, chunker: SemanticChunker) -> List[Document]:
    """Carga un PDF y lo divide en chunks semánticos con metadatos enriquecidos."""
    try:
        # Cargar PDF
        loader = PyPDFLoader(str(pdf_path))
        pages  = loader.load()
        
        # Concatenar texto del documento completo
        full_text = ' '.join([p.page_content for p in pages])
        
        # Aplicar chunking semántico
        raw_chunks = chunker.create_documents([full_text])
        
        # Enriquecer metadatos de cada chunk
        doc_id = pdf_path.stem  # nombre del archivo sin extensión
        enriched_chunks = []
        for i, chunk in enumerate(raw_chunks):
            chunk.metadata = {
                'doc_id'    : doc_id,
                'source'    : str(pdf_path),
                'chunk_id'  : f'{doc_id}_chunk_{i:03d}',
                'chunk_idx' : i,
                'total_chunks': len(raw_chunks),
                'page_count': len(pages),
                'char_count': len(chunk.page_content)
            }
            enriched_chunks.append(chunk)
        
        return enriched_chunks
    except Exception as e:
        print(f'⚠️  Error procesando {pdf_path.name}: {e}')
        return []


# ── Procesar todos los PDFs ───────────────────────────────────────────────
all_chunks   = []
chunk_stats  = []

print(f'Procesando {len(pdfs)} documentos con SemanticChunker...')
for i, pdf in enumerate(pdfs):
    chunks = load_and_chunk_document(pdf, semantic_chunker)
    chunk_stats.append({'doc': pdf.name, 'n_chunks': len(chunks)})
    all_chunks.extend(chunks)
    if (i+1) % 10 == 0:
        print(f'  [{i+1}/{len(pdfs)}] {pdf.name} → {len(chunks)} chunks')

print(f'\n✅ Procesamiento completado')
print(f'   Total de chunks semánticos: {len(all_chunks)}')
print(f'   Promedio de chunks por doc: {len(all_chunks)/len(pdfs):.1f}')

In [ ]:
# ── Análisis de los chunks ────────────────────────────────────────────────
import statistics

chunk_lengths = [len(c.page_content) for c in all_chunks]

print('📊 Estadísticas de chunks semánticos:')
print(f'   Mínimo:  {min(chunk_lengths):,} chars')
print(f'   Máximo:  {max(chunk_lengths):,} chars')
print(f'   Promedio: {statistics.mean(chunk_lengths):,.0f} chars')
print(f'   Mediana:  {statistics.median(chunk_lengths):,.0f} chars')
print(f'   Desv. estándar: {statistics.stdev(chunk_lengths):,.0f} chars')

print('\n📄 Ejemplo de chunk semántico:')
print('-' * 60)
example = all_chunks[0]
print(f'Doc ID:    {example.metadata["doc_id"]}')
print(f'Chunk ID:  {example.metadata["chunk_id"]}')
print(f'Chars:     {example.metadata["char_count"]}')
print(f'Texto:     {example.page_content[:300]}...')

## 5 · Comparación: Fixed-size vs Semántico

Demostramos la diferencia entre el chunking de tamaño fijo (P1) y el chunking semántico (P2).

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Chunking fijo (Práctica 1) ────────────────────────────────────────────
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Usar el primer PDF como ejemplo
test_pdf = pdfs[0]
loader   = PyPDFLoader(str(test_pdf))
pages    = loader.load()
full_text = ' '.join([p.page_content for p in pages])

# Fixed chunks
fixed_chunks = fixed_splitter.create_documents([full_text])

# Semantic chunks
semantic_chunks_test = semantic_chunker.create_documents([full_text])

print(f'📄 Documento: {test_pdf.name}')
print(f'   Texto total: {len(full_text):,} caracteres')
print()
print('⚡ Chunking de tamaño fijo (P1):')
print(f'   Número de chunks: {len(fixed_chunks)}')
print(f'   Tamaño promedio: {sum(len(c.page_content) for c in fixed_chunks)/len(fixed_chunks):.0f} chars')
print()
print('🧠 Chunking semántico (P2):')
print(f'   Número de chunks: {len(semantic_chunks_test)}')
print(f'   Tamaño promedio: {sum(len(c.page_content) for c in semantic_chunks_test)/len(semantic_chunks_test):.0f} chars')
print()
print('💡 Ventaja del chunking semántico:')
print('   • Preserva unidades temáticas completas')
print('   • Evita cortar ideas a la mitad')
print('   • Mayor coherencia en la recuperación')
print('   • Mejor relevancia en las respuestas RAG')

## 6 · Indexar en FAISS

In [ ]:
# ── Construir índice FAISS con chunks semánticos ──────────────────────────
# Se procesan en lotes para evitar límites de tasa de la API
BATCH_SIZE = 50
faiss_index = None

print(f'Construyendo índice FAISS con {len(all_chunks)} chunks semánticos...')
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    if faiss_index is None:
        faiss_index = FAISS.from_documents(batch, embeddings_model)
    else:
        faiss_index.add_documents(batch)
    
    progress = min(i + BATCH_SIZE, len(all_chunks))
    print(f'  [{progress}/{len(all_chunks)}] vectores indexados')
    time.sleep(1)  # respetar rate limit

print(f'\n✅ Índice FAISS construido')
print(f'   Vectores totales: {faiss_index.index.ntotal}')

In [ ]:
# ── Guardar índice FAISS ──────────────────────────────────────────────────
FAISS_PATH = INDEX_DIR / 'faiss_semantic'
faiss_index.save_local(str(FAISS_PATH))
print(f'✅ Índice FAISS guardado en: {FAISS_PATH}')

# También guardar los chunks como pickle para referencia
with open(INDEX_DIR / 'semantic_chunks.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
print(f'✅ Chunks semánticos guardados: {INDEX_DIR}/semantic_chunks.pkl')

In [ ]:
# ── Test de recuperación ──────────────────────────────────────────────────
test_query = 'What are the genetic mutations associated with breast cancer?'
results = faiss_index.similarity_search_with_score(test_query, k=3)

print(f'🔍 Query: "{test_query}"\n')
for i, (doc, score) in enumerate(results):
    print(f'Resultado {i+1} (score={score:.4f}):')
    print(f'  Doc: {doc.metadata["doc_id"]}')
    print(f'  Chunk: {doc.metadata["chunk_id"]}')
    print(f'  Texto: {doc.page_content[:200]}...')
    print()

## Resumen

| Métrica | Valor |
|---|---|
| Documentos procesados | 50 |
| Chunking | SemanticChunker (percentile 95) |
| Modelo de embeddings | text-embedding-004 |
| Índice | FAISS (cosine similarity) |
| Uso en el sistema | Base para recuperación MMR y KG-RAG |

**Próximos pasos:** El índice FAISS construido aquí será utilizado en:
- `P2_02_query_transformation.ipynb` → para probar HyDE y Query Decomposition
- `P2_03_advanced_retrieval.ipynb` → para búsqueda MMR
- `RAG_KnowledgeGraph_Completo.ipynb` → sistema integrado completo